In [ ]:
import numpy as np
import pandas as pd
import torch
import os
import random
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import copy

from transformers import AutoModel, AutoTokenizer
from torch.nn import functional as F

from sklearn import metrics
from tqdm.auto import tqdm, trange

import torch.optim as optim
import math
from torch.optim.lr_scheduler import _LRScheduler

from torch.utils.data import TensorDataset
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

In [ ]:
def set_seed(seed: int = 42) :
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Random seed set as {seed}")
    
seed = 42
set_seed(seed)

device = torch.device("cuda:0")

In [ ]:
train_df = pd.read_csv('./pretraining_data/train_profile.csv')
valid_df = pd.read_csv('./pretraining_data/valid_profile.csv')
test_df = pd.read_csv('./pretraining_data/test_profile.csv')

In [ ]:
train_df

In [ ]:
train_df.columns[2:]

In [ ]:
train_profile = train_df[train_df.columns[2:]].values
valid_profile = valid_df[valid_df.columns[2:]].values
test_profile = test_df[test_df.columns[2:]].values

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('DeepChem/ChemBERTa-77M-MTR')

In [ ]:
class SmilesAdmetDataset(Dataset):
    def __init__(self, smiles_embeddings, admet_profiles):
        """
        smiles_embeddings: List[Tensor] or np.ndarray, shape (N, E) - E: embedding dim
        admet_profiles: List[Tensor] or np.ndarray, shape (N, 39)
        """
        assert len(smiles_embeddings) == len(admet_profiles), "Dimensionality invalid!"
        
        self.smiles_embeddings = torch.tensor(smiles_embeddings, dtype=torch.float32)
        self.admet_profiles = torch.tensor(admet_profiles, dtype=torch.float32)

    def __len__(self):
        return len(self.smiles_embeddings)

    def __getitem__(self, idx):
        return {
            "smiles_emb": self.smiles_embeddings[idx],  
            "admet_vec": self.admet_profiles[idx]       
        }

def prepare_input(tokenizer, smiles):
    inputs = tokenizer(smiles, add_special_tokens=True, truncation=True, max_length=512, padding="max_length")
    for k, v in inputs.items():
        inputs[k] = torch.tensor(v, dtype=torch.long)
    return inputs

class ChemiDataset(Dataset):
    def __init__(self, smiles, profiles):
        self.smiles = smiles
        self.label =  torch.tensor(profiles, dtype=torch.float32)
            
    def __len__(self):
        return len(self.smiles)
    
    def __getitem__(self, item):
        inputs = prepare_input(tokenizer, self.smiles[item])
        label = self.label[item]
        return {
            "smiles_emb": inputs,  
            "admet_vec": label       
        }

In [ ]:
def get_dataloader(smiles, admet_profiles, batch_size=64, shuffle=True, num_workers=1, drop_last = False):
    dataset = ChemiDataset(smiles, admet_profiles)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, drop_last = drop_last)
    return dataloader

In [ ]:
tr_loader = get_dataloader(train_df['smiles'].values, train_profile, batch_size=128, drop_last = True)
va_loader = get_dataloader(valid_df['smiles'].values, valid_profile, batch_size=128, shuffle = True)

In [ ]:
class DrugEncoder(nn.Module):
    def __init__(self, pretrained_model='DeepChem/ChemBERTa-77M-MTR', hidden_dim = 384):
        super(DrugEncoder, self).__init__()
        self.bert = AutoModel.from_pretrained(pretrained_model)
       
    def forward(self, inputs):
        embedding = self.bert(**inputs)
        embedding = embedding[0]
        hidden = embedding[:, 0, :]

        return hidden

class ProfileEncoder(nn.Module):
    def __init__(self, input_dim = 128, hidden_dim = 128):
        super(ProfileEncoder, self).__init__()
        
        self.sub_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim))

    def forward(self, input_vec):
        x = self.sub_layer(input_vec)
        return x 

In [ ]:
class ADMET_FM_pretrain(nn.Module):
    def __init__(self, drug_input_dim=128, drug_hidden_dim = 128, profile_input_dim=39, profile_hidden_dim=128, init_temperature=0.07):
        
        super(ADMET_FM_pretrain, self).__init__()
        
        self.drug_encoder = DrugEncoder(hidden_dim = drug_hidden_dim)
        self.profile_encoder = ProfileEncoder(input_dim = profile_input_dim, hidden_dim = profile_hidden_dim)
        
        self.drug_projection = nn.Linear(drug_hidden_dim, profile_hidden_dim)
        self.profile_projection = nn.Linear(profile_hidden_dim, profile_hidden_dim)
        
        self.profile_recon = nn.Sequential(nn.Linear(drug_hidden_dim, profile_input_dim), nn.Sigmoid())
        
    def forward(self, drug_input, profile_input):
        drug_feat = self.drug_encoder(drug_input)         
        profile_feat = self.profile_encoder(profile_input) 

        drug_proj = self.drug_projection(drug_feat)
        profile_proj = self.profile_projection(profile_feat)
        
        profile_recon = self.profile_recon(drug_feat)

        return drug_feat, profile_feat, drug_proj, profile_proj, profile_recon

In [ ]:
class CWCLoss(nn.Module):
    def __init__(self, alpha=0.5, temperature=0.07):
        super().__init__()
        self.alpha = alpha
        self.temperature = temperature
        
    def forward(self, proj_u, proj_v, input_v):
        
        batch_size = proj_u.size(0)
        labels = torch.arange(batch_size, device=proj_u.device)

        p_norm = F.normalize(proj_u, dim=-1)
        q_norm = F.normalize(proj_v, dim=-1)

        logits_p_q = torch.matmul(p_norm, q_norm.T) / self.temperature
        loss_v_u = F.cross_entropy(logits_p_q.T, labels)
       
        weights_v = torch.matmul(q_norm, q_norm.T)
        profile_input_norm = F.normalize(input_v, dim=-1)
        weights_v = torch.matmul(profile_input_norm, profile_input_norm.T)   
        weights_v = (weights_v / 2) + 0.5
        soft_labels_v = F.normalize(weights_v, p=1, dim=-1)
        loss_u_v = F.cross_entropy(logits_p_q, soft_labels_v)

        loss = self.alpha * loss_u_v + (1 - self.alpha) * loss_v_u
        
        return loss, loss_u_v, loss_v_u

In [ ]:
def train(model, train_dataloader, val_dataloader, loss_fn, loss_recon, optimizer, scheduler, device, epochs=10, model_dir=None):
    
    model.train()
    model.to(device)

    train_loss_list = []
    val_loss_list = []
    best_val_loss = float('inf') 

    save_dir = model_dir
    save_path = os.path.join(save_dir, 'best_model_state.pth')
    
    if not os.path.exists(save_dir):
        os.makedirs(save_dir, exist_ok=True)
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        for num, batch in enumerate(train_dataloader):
            drug_input = batch["smiles_emb"].to(device)
            profile_input = batch["admet_vec"].to(device)

            optimizer.zero_grad()
            
            drug_feat, profile_feat, drug_proj, profile_proj, profile_recon = model(drug_input, profile_input)
            loss, loss_drug, loss_profile = loss_fn(drug_proj, profile_proj, profile_input)
            loss2 = loss_recon(profile_recon, profile_input)
            
            total_loss = loss + loss2
            total_loss.backward()
                
            optimizer.step()
            scheduler.step()           
            total_train_loss += (loss.item() + loss2.item())

        avg_train_loss = total_train_loss / len(train_dataloader)
        train_loss_list.append(avg_train_loss)

        model.eval() 
        total_val_loss = 0
        with torch.no_grad(): 
            for batch in val_dataloader:
                drug_input = batch["smiles_emb"].to(device)
                profile_input = batch["admet_vec"].to(device)
                
                drug_feat, profile_feat, drug_proj, profile_proj, profile_recon = model(drug_input, profile_input)
                val_loss, loss_, loss__ = loss_fn(drug_proj, profile_proj, profile_input)
                val_loss2 = loss_recon(profile_recon, profile_input)
                
                total_val_loss += (val_loss.item() + val_loss2.item())

        avg_val_loss = total_val_loss / len(val_dataloader)
        val_loss_list.append(avg_val_loss)

        print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), save_path)
            print(f"  >> New best model saved with Val Loss: {best_val_loss:.4f}")
        
    return train_loss_list, val_loss_list

In [ ]:
model = ADMET_FM_pretrain(drug_input_dim=384, drug_hidden_dim = 384, profile_input_dim=train_profile.shape[1], profile_hidden_dim=384)
model = model.to(device)

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

epochs = 100
total_steps = epochs * len(tr_loader)
warmup_epochs = 10 
lr = 3e-4

optimizer = optim.AdamW(model.parameters(), lr = lr)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=lr,              
    total_steps=total_steps,
    pct_start=0.1,          
    anneal_strategy='cos',  
    div_factor=300,         
    final_div_factor=30   
)

In [ ]:
loss_fn = CWCLoss(alpha = 0.9, temperature=0.1)
loss_recon = nn.MSELoss()

In [ ]:
train_losses, val_losses = train(model, tr_loader, va_loader, loss_fn, loss_recon, optimizer, scheduler, 
                                 device, epochs=epochs, model_dir='./pre_trained_weights/' )

In [ ]:
from typing import Dict

def get_embeddings(final_model, data_loader, device) -> Dict[str, np.ndarray]:
    final_model.eval()

    drug_feat_list, profile_feat_list = [], []
    drug_proj_list, profile_proj_list = [], []

    with torch.no_grad():
        for batch in data_loader:
            drug_input = batch["smiles_emb"].to(device)
            profile_input = batch["admet_vec"].to(device)

            drug_feat, profile_feat, drug_proj, profile_proj, _ = final_model(drug_input, profile_input)

            drug_feat_list.append(drug_feat)
            profile_feat_list.append(profile_feat)
            drug_proj_list.append(drug_proj)
            profile_proj_list.append(profile_proj)

    return {
        "drug_feat": torch.cat(drug_feat_list).cpu().numpy(),
        "profile_feat": torch.cat(profile_feat_list).cpu().numpy(),
        "drug_proj": torch.cat(drug_proj_list).cpu().numpy(),
        "profile_proj": torch.cat(profile_proj_list).cpu().numpy(),
    }

In [ ]:
test_loader = get_dataloader(test_df['smiles'].values, test_profile, batch_size=128, shuffle = False)

In [ ]:
final_model = ADMET_FM_pretrain(drug_input_dim=384, drug_hidden_dim = 384, profile_input_dim=train_profile.shape[1], profile_hidden_dim=384)
final_model.to(device)

final_state = torch.load('./pre_trained_weights/best_model_state.pth', weights_only = True)
final_model.load_state_dict(final_state)

In [ ]:
test_embed_dict = get_embeddings(final_model, test_loader, device)

In [ ]:
def topk_retrieval_accuracy(compound_embeddings: np.ndarray,
                            admet_embeddings: np.ndarray,
                            topk=(1, 5, 10)):
    
    comp_norm = compound_embeddings / np.linalg.norm(compound_embeddings, axis=1, keepdims=True)
    admet_norm = admet_embeddings / np.linalg.norm(admet_embeddings, axis=1, keepdims=True)
    
    sim_matrix = np.dot(comp_norm, admet_norm.T)
    
    n = sim_matrix.shape[0]
    correct_indices = np.arange(n)   
    ranks = np.argsort(-sim_matrix, axis=1) 

    results = {}
    for k in topk:
        topk_hits = np.any(ranks[:, :k] == correct_indices[:, None], axis=1)
        acc = np.mean(topk_hits)
        results[k] = acc
    
    return results

In [ ]:
d2p_acc = topk_retrieval_accuracy(test_embed_dict['drug_proj'], test_embed_dict['profile_proj'], topk=(1, 3, 5, 10, 30, 50))
p2d_acc = topk_retrieval_accuracy(test_embed_dict['profile_proj'], test_embed_dict['drug_proj'], topk=(1, 3, 5, 10, 30, 50))

In [ ]:
k_values = list(d2p_acc.keys())
x_indices = np.arange(len(k_values))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparisons = [
    ('Drug to Profile', d2p_acc, '#d62728'),
    ('Profile to Drug', p2d_acc, '#1f77b4'),
]
for ax, (title, data, color) in zip(axes, comparisons):
    metrics = list(data.values())
    ax.plot(x_indices, metrics,
            marker='s', markersize=8, linewidth=2, color=color)
    ax.set_xticks(x_indices)
    ax.set_xticklabels(k_values)
    ax.set_xlabel('Number of retrievals (K)', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'Performance by Retrieval Count (K) — {title}', fontsize=13, pad=12)
    y_min, y_max = min(metrics), max(metrics)
    margin = (y_max - y_min) * 0.1
    ax.set_ylim(y_min - margin, y_max + margin)
    for i, v in enumerate(metrics):
        ax.annotate(f'{v:.3f}', (x_indices[i], v),
                    textcoords="offset points", xytext=(0, 10),
                    ha='center', fontsize=8, color='black')
plt.tight_layout()
plt.show()

In [ ]:
from typing import Dict

def get_recons(final_model, data_loader, device) -> Dict[str, np.ndarray]:
    final_model.eval()

    recon_list = []

    with torch.no_grad():
        for batch in data_loader:
            drug_input = batch["smiles_emb"].to(device)
            profile_input = batch["admet_vec"].to(device)

            drug_feat, profile_feat, drug_proj, profile_proj, profile_recon = final_model(drug_input, profile_input)

            recon_list.append(profile_recon)

    return torch.cat(recon_list).cpu().numpy()

In [ ]:
test_recons = get_recons(final_model, test_loader, device)

In [ ]:
import numpy as np
from scipy.stats import pearsonr

rmse_per_column = np.sqrt(np.mean((test_profile - test_recons)**2, axis=0))

pcc_per_column = np.array([
    pearsonr(test_profile[:, i], test_recons[:, i])[0] 
    for i in range(test_profile.shape[1])
])

for i in range(test_profile.shape[1]):
    print(f"Column {i:02d} | PCC: {pcc_per_column[i]:.4f} | RMSE: {rmse_per_column[i]:.4f}")

In [ ]:
mean_pcc = np.mean(pcc_per_column)
std_pcc  = np.std(pcc_per_column)

print(f"Mean PCC: {mean_pcc:.4f} ± {std_pcc:.4f}")